### Source Tables:
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3order` — medication orders (TypeCode = 'Medication')
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension` — medication details (dose, route, refills)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem` — drug formulary with RxNorm codes

### Strategy:
- Join dbo_cv3order (Medication orders) with dbo_cv3medicationextension via GUID
- Join to dbo_sxammgenericitem via PrescriptionGenericItemID for RxNorm codes
- Map RxNormCode to OMOP drug_concept_id via domain_source_to_concept
- Map OrderRouteCode via domain_source_to_concept
- Use drug_type_concept_id = 32817 (EHR)
- Calculate drug_exposure_end_date from StopDtm or fallback to start date

### Notes:
- This notebook depends on source_to_person being populated for allscripts_scm
- provider_id and visit_occurrence_id will be NULL until those mapping tables are created
- RxNorm concept mapping requires OMOP vocabulary tables to be loaded
- dbo_cv3order links to patients via ClientGUID → cv3client.GUID

# EDA

In [0]:
# %sql
# -- EDA: TypeCode distribution (confirm 'Medication' is the correct filter)
# SELECT TypeCode, COUNT(*) as cnt
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order
# WHERE Active = TRUE
# GROUP BY TypeCode
# ORDER BY cnt DESC

In [0]:
# %sql
# -- EDA: Medication order count and MedExt join rate
# SELECT
#   COUNT(*) AS total_med_orders,
#   SUM(CASE WHEN medext.GUID IS NOT NULL THEN 1 ELSE 0 END) AS with_medext,
#   ROUND(100.0 * SUM(CASE WHEN medext.GUID IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS medext_pct,
#   SUM(CASE WHEN medext.PrescriptionGenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS with_generic_item_id,
#   ROUND(100.0 * SUM(CASE WHEN medext.PrescriptionGenericItemID IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS generic_item_pct,
#   COUNT(DISTINCT ord.ClientGUID) AS distinct_patients
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
# LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
#   ON medext.GUID = ord.GUID
#   AND medext.Active = TRUE
# WHERE ord.Active = TRUE
#   AND ord.TypeCode = 'Medication'

In [0]:
# %sql
# -- EDA: RxNorm coverage in sxammgenericitem
# SELECT
#   COUNT(*) AS total_generic_items,
#   SUM(CASE WHEN RxNormCode IS NOT NULL AND TRIM(RxNormCode) != '' THEN 1 ELSE 0 END) AS with_rxnorm,
#   ROUND(100.0 * SUM(CASE WHEN RxNormCode IS NOT NULL AND TRIM(RxNormCode) != '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS rxnorm_pct
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem
# WHERE Active = TRUE

In [0]:
# %sql
# -- EDA: RxNorm → OMOP concept match rate
# SELECT
#   COUNT(DISTINCT gi.RxNormCode) AS distinct_rxnorm_codes,
#   COUNT(DISTINCT CASE WHEN c.concept_id IS NOT NULL THEN gi.RxNormCode END) AS matched,
#   COUNT(DISTINCT CASE WHEN c.concept_id IS NULL THEN gi.RxNormCode END) AS unmatched,
#   ROUND(100.0 * COUNT(DISTINCT CASE WHEN c.concept_id IS NOT NULL THEN gi.RxNormCode END) / COUNT(DISTINCT gi.RxNormCode), 2) AS match_pct
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
# LEFT JOIN _exponent.omop.concept c
#   ON c.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
#   AND c.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
# WHERE gi.Active = TRUE
#   AND gi.RxNormCode IS NOT NULL
#   AND TRIM(CAST(gi.RxNormCode AS STRING)) != ''

In [0]:
# %sql
# -- EDA: Date field population
# SELECT
#   COUNT(*) AS total_med_orders,
#   SUM(CASE WHEN RequestedDtm IS NOT NULL THEN 1 ELSE 0 END) AS has_requested_dtm,
#   SUM(CASE WHEN Entered IS NOT NULL THEN 1 ELSE 0 END) AS has_entered,
#   SUM(CASE WHEN CreatedWhen IS NOT NULL THEN 1 ELSE 0 END) AS has_created_when,
#   SUM(CASE WHEN StopDtm IS NOT NULL THEN 1 ELSE 0 END) AS has_stop_dtm,
#   SUM(CASE WHEN COALESCE(RequestedDtm, Entered, CreatedWhen) IS NOT NULL THEN 1 ELSE 0 END) AS has_any_start_date
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order
# WHERE Active = TRUE
#   AND TypeCode = 'Medication'

In [0]:
# %sql
# -- EDA: Route code distribution
# SELECT OrderRouteCode, COUNT(*) AS cnt
# FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension
# WHERE Active = TRUE
#   AND OrderRouteCode IS NOT NULL
# GROUP BY OrderRouteCode
# ORDER BY cnt DESC
# LIMIT 30

# Concept Mappings

In [0]:
%sql
-- Populate domain_source_to_concept: Drug mappings (RxNorm → OMOP concept)
-- Maps GenericItemID → RxNormCode → concept_id
INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
    source_system,
    source_table,
    source_field,
    domain_id,
    source_id,
    source_value,
    omop_concept_id,
    active_flag,
    last_update_tsp
)
SELECT DISTINCT
    'allscripts_scm' AS source_system,
    'dbo_sxammgenericitem' AS source_table,
    'RxNormCode' AS source_field,
    'Drug' AS domain_id,
    TRIM(CAST(gi.RxNormCode AS STRING)) AS source_id,
    gi.GenericItemName AS source_value,
    COALESCE(std_c.concept_id, c.concept_id) AS omop_concept_id,
    1 AS active_flag,
    CURRENT_TIMESTAMP() AS last_update_tsp
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
INNER JOIN _exponent.omop.concept c
    ON c.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
    AND c.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
-- Resolve non-standard concepts via 'Maps to'
LEFT JOIN _exponent.omop.concept_relationship cr
    ON cr.concept_id_1 = c.concept_id
    AND cr.relationship_id = 'Maps to'
LEFT JOIN _exponent.omop.concept std_c
    ON std_c.concept_id = cr.concept_id_2
    AND std_c.standard_concept = 'S'
    AND std_c.domain_id = 'Drug'
WHERE gi.Active = TRUE
    AND gi.RxNormCode IS NOT NULL
    AND TRIM(CAST(gi.RxNormCode AS STRING)) != ''
    -- Avoid duplicates if already mapped
    AND NOT EXISTS (
        SELECT 1 FROM _exponent.omop_mapping.domain_source_to_concept existing
        WHERE existing.source_system = 'allscripts_scm'
          AND existing.domain_id = 'Drug'
          AND existing.source_id = TRIM(CAST(gi.RxNormCode AS STRING))
    )

In [0]:
# %sql
# -- Insert Route mappings for SCM
# -- Run this AFTER reviewing EDA route distribution above
# -- Match source OrderRouteCode values to OMOP standard Route concepts
# INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
#     source_system, source_table, source_field, domain_id,
#     source_id, source_value, omop_concept_id, active_flag, last_update_tsp
# )
# VALUES
# -- Update these based on EDA route distribution results
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Oral', 'Oral', 4132161, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Intravenous', 'Intravenous', 4171047, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Subcutaneous', 'Subcutaneous', 4142048, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Intramuscular', 'Intramuscular', 4302612, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Topical', 'Topical', 4263689, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Ophthalmic', 'Ophthalmic', 4184451, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Rectal', 'Rectal', 4115462, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Nasal', 'Nasal', 4262914, 1, CURRENT_TIMESTAMP()),
# ('allscripts_scm', 'dbo_cv3medicationextension', 'OrderRouteCode', 'Route', 'Inhalation', 'Inhalation', 4120036, 1, CURRENT_TIMESTAMP());

In [0]:
# %sql
# -- Verify domain_source_to_concept for SCM drugs
# SELECT source_system, domain_id, COUNT(*) as cnt
# FROM _exponent.omop_mapping.domain_source_to_concept
# WHERE source_system = 'allscripts_scm'
# GROUP BY source_system, domain_id

# Transformation

In [0]:
%sql
-- Create silver_drug_exposure temp view for SCM
CREATE OR REPLACE TEMPORARY VIEW silver_drug_exposure AS
SELECT
  -- Drug concept mapping via domain_source_to_concept
  COALESCE(drug_concept.omop_concept_id, 0) AS drug_concept_id,

  -- Start date
  DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)) AS drug_exposure_start_date,
  COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS drug_exposure_start_datetime,

  -- End date: explicit StopDtm or fallback to start date
  CASE
    WHEN ord.StopDtm IS NOT NULL THEN DATE(ord.StopDtm)
    ELSE DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen))
  END AS drug_exposure_end_date,

  CASE
    WHEN ord.StopDtm IS NOT NULL THEN ord.StopDtm
    ELSE COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
  END AS drug_exposure_end_datetime,

  -- Verbatim end date (only if explicit StopDtm)
  CASE WHEN ord.StopDtm IS NOT NULL THEN DATE(ord.StopDtm) ELSE NULL END AS verbatim_end_date,

  -- Drug type concept: 32817 = EHR
  32817 AS drug_type_concept_id,

  NULL AS stop_reason,
  medext.NumRefills AS refills,
  TRY_CAST(medext.DispenseAmount AS DOUBLE) AS quantity,
  NULL AS days_supply,
  medext.RxInstructions AS sig,
  COALESCE(route_concept.omop_concept_id, 0) AS route_concept_id,
  NULL AS lot_number,

  -- Source values
  COALESCE(medext.OrderedAsDisplay, gi.GenericItemName, ord.Name) AS drug_source_value,
  0 AS drug_source_concept_id,
  medext.OrderRouteCode AS route_source_value,
  COALESCE(medext.DosageLow, medext.Uom) AS dose_unit_source_value,

  -- FK source values (CHR(31) separated, 4-segment format)
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING)) AS person_source_value,

  CASE
    WHEN ord.CareProviderGUID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3careprovider', 'GUID', CAST(ord.CareProviderGUID AS STRING))
    ELSE NULL
  END AS provider_source_value,

  CASE
    WHEN ord.ClientVisitGUID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3clientvisit', 'GUID', CAST(ord.ClientVisitGUID AS STRING))
    ELSE NULL
  END AS visit_occurrence_source_value,

  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3order', 'GUID', CAST(ord.GUID AS STRING)) AS drug_exposure_source_value,
  'allscripts_scm' AS source_system

FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord

INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
  AND medext.Active = TRUE

LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON gi.GenericItemID = medext.PrescriptionGenericItemID
  AND gi.Active = TRUE

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
  AND stp.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.domain_source_to_concept drug_concept
  ON drug_concept.source_id = TRIM(CAST(gi.RxNormCode AS STRING))
  AND drug_concept.domain_id = 'Drug'
  AND drug_concept.source_system = 'allscripts_scm'

LEFT JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.source_id = medext.OrderRouteCode
  AND route_concept.domain_id = 'Route'
  AND route_concept.source_system = 'allscripts_scm'

WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL

In [0]:
# %sql
# -- Preview silver temp view
# SELECT * FROM silver_drug_exposure LIMIT 10

In [0]:
# %sql
# -- Check for duplicates on merge key
# SELECT drug_exposure_source_value, drug_type_concept_id, COUNT(*) as cnt
# FROM silver_drug_exposure
# GROUP BY drug_exposure_source_value, drug_type_concept_id
# HAVING COUNT(*) > 1
# LIMIT 20

In [0]:
# %sql
# -- Validation: concept mapping rates
# SELECT
#   COUNT(*) AS total_records,
#   SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_drug,
#   ROUND(100.0 * SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_drug_pct,
#   SUM(CASE WHEN route_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_route,
#   ROUND(100.0 * SUM(CASE WHEN route_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_route_pct,
#   COUNT(DISTINCT person_source_value) AS distinct_patients,
#   MIN(drug_exposure_start_date) AS min_start_date,
#   MAX(drug_exposure_start_date) AS max_start_date
# FROM silver_drug_exposure

# Write to Silver

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING (
  SELECT * FROM (
    SELECT 
      *,
      ROW_NUMBER() OVER (
        PARTITION BY drug_exposure_source_value, drug_type_concept_id 
        ORDER BY drug_exposure_start_date DESC
      ) AS rn
    FROM silver_drug_exposure
  ) WHERE rn = 1
) AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value 
   AND t.drug_type_concept_id = s.drug_type_concept_id

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id             = s.drug_concept_id,
  t.drug_exposure_start_date    = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date      = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime  = s.drug_exposure_end_datetime,
  t.verbatim_end_date           = s.verbatim_end_date,
  t.drug_type_concept_id        = s.drug_type_concept_id,
  t.stop_reason                 = s.stop_reason,
  t.refills                     = s.refills,
  t.quantity                    = s.quantity,
  t.days_supply                 = s.days_supply,
  t.sig                         = s.sig,
  t.route_concept_id            = s.route_concept_id,
  t.lot_number                  = s.lot_number,
  t.drug_source_value           = s.drug_source_value,
  t.drug_source_concept_id      = s.drug_source_concept_id,
  t.route_source_value          = s.route_source_value,
  t.dose_unit_source_value      = s.dose_unit_source_value,
  t.person_source_value         = s.person_source_value,
  t.provider_source_value       = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value   = s.visit_detail_source_value,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver layer
# SELECT * FROM _exponent.omop_silver.drug_exposure
# WHERE source_system = 'allscripts_scm'
# LIMIT 10

# Register Drug Exposure IDs

In [0]:
%sql
-- Insert new mappings to source_to_drug_exposure
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
    source_system,
    drug_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.drug_exposure_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.drug_exposure
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

# Write to Gold

In [0]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.drug_exposure AS gold
MERGE INTO _exponent.omop_scm.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.drug_concept_id              = src.drug_concept_id,
  gold.drug_exposure_start_date     = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date       = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
  gold.verbatim_end_date            = src.verbatim_end_date,
  gold.drug_type_concept_id         = src.drug_type_concept_id,
  gold.stop_reason                  = src.stop_reason,
  gold.refills                      = src.refills,
  gold.quantity                     = src.quantity,
  gold.days_supply                  = src.days_supply,
  gold.sig                          = src.sig,
  gold.route_concept_id             = src.route_concept_id,
  gold.lot_number                   = src.lot_number,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.drug_source_value            = src.drug_source_value,
  gold.drug_source_concept_id       = src.drug_source_concept_id,
  gold.route_source_value           = src.route_source_value,
  gold.dose_unit_source_value       = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

# Validation

In [0]:
# %sql
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.drug_exposure WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_drug_exposure WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.drug_exposure

In [0]:
# %sql
# SELECT 
#   COUNT(*) AS total_records,
#   SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_count,
#   ROUND(100.0 * SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_pct
# FROM _exponent.omop.drug_exposure

In [0]:
# %sql
# SELECT * FROM _exponent.omop.drug_exposure
# WHERE drug_concept_id != 0
# LIMIT 10